[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/39.%20The%20Channel%20Dredging%20%26%20Deepening%20Investment%20Problem/P39-Tier-7_executed.ipynb)

---


# 39. The Channel Dredging and Deepening Investment Problem

## Tier 7: Human–AI Symbiotic Partnership

### Goal

This notebook builds an illustrative simulation of collaboration between an explainable AI recommendation system and human channel experts. It represents a 180-day partnership in which the AI proposes actions for channel segments, an expert reviews each proposal, and trust metrics are updated from the interaction.

The notebook's computational method is a sequential, stochastic simulation. It is not a dredging investment optimizer: it does not choose an investment portfolio, minimize cost, or maximize project value. Its purpose is to make the human–AI decision process and its measured indicators explicit.

### Inputs, outputs, and scope

**Inputs** include the segment definitions, expert attributes, partnership duration, and simulated depth, traffic, weather, and sensor conditions. **Outputs** include AI recommendations, simulated human decisions, trust and collaboration metrics, and diagnostic plots.

**Model assumptions:** Condition data and human decisions are generated by simple probability rules for teaching purposes. Trust changes according to whether a recommendation is accepted, modified, or rejected; this is a proxy, not an empirically validated trust model. The decision-quality figures are assumed scenario rates, not measured outcomes. Although decisions and feedback are recorded for analysis, the AI recommendation rules and confidence formula do not learn or adapt during the simulation. The model does not represent dredging costs, engineering constraints, environmental permits, realized project outcomes, or a comparison against human-only and AI-only baselines.

### How the simulation works

For each day and segment, the simulation generates conditions, obtains a rule-based AI recommendation, selects an expert, simulates that expert's response, and updates the expert's trust. The trust level is a **state**: it summarizes the simulated relationship at a point in time and affects later expert selection and acceptance probabilities. Because this state changes after each interaction, the order of decisions matters.

Let $c$ be AI confidence, $t$ the selected expert's trust, and $e$ the expertise multiplier. The simulated acceptance probability is

$$p = \min(1, cte).$$

For a uniform random draw $u \in [0,1)$, the response is accept when $u < 0.7p$, modify when $0.7p \leq u < 0.9p$, and reject otherwise. The factors 0.7 and 0.9 are simulation assumptions that allocate probability to the three response categories; they are not estimates from operational data.

After a response, trust is bounded to $[0.1,1]$: acceptance adds $0.02$, rejection subtracts $0.01$, and modification leaves trust unchanged. In code, these rules appear in `simulate_human_decision` and `update_trust_metrics`. Random condition and response draws make results stochastic; a run can therefore vary unless a random seed is set.

### What to look for

- How recommendation confidence, expert trust, and expertise influence simulated responses.
- How the trust update rule changes later interactions and expert selection.
- The distinction between descriptive simulation metrics and measured operational performance.
- Which conclusions are supported by the coded rules, and which would require real data and outcome validation.

In [ ]:
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

In [ ]:
@dataclass
class HumanExpert:
    """A domain expert with experience, trust, and availability attributes."""

    expert_id: str
    name: str
    experience_years: int
    expertise_areas: List[str]
    trust_level: float
    decision_accuracy: float
    availability: float
    feedback_quality: float


@dataclass
class AIRecommendation:
    """An AI proposal with a timestamp, confidence, and explanation."""

    recommendation_id: str
    timestamp: datetime
    decision_type: str
    segment_id: int
    action: str
    confidence: float
    reasoning: List[str]
    data_sources: List[str]
    uncertainty_factors: List[str]
    expected_outcome: Dict[str, float]


@dataclass
class HumanDecision:
    """A simulated expert response to one AI recommendation."""

    decision_id: str
    expert_id: str
    recommendation_id: str
    timestamp: datetime
    action: str
    reasoning: str
    confidence: float
    time_spent_minutes: int
    additional_context: Dict[str, Any]


@dataclass
class TrustMetrics:
    """Recorded measures associated with a human–AI interaction."""

    timestamp: datetime
    expert_id: str
    ai_trust_score: float
    recommendation_trust: float
    explanation_trust: float
    overall_satisfaction: float
    collaboration_effectiveness: float


# Partnership timing is shared by the simulation and its result summaries.
partnership_start = datetime(2024, 1, 1, 8, 0, 0)
partnership_duration = timedelta(days=180)
decision_interval = timedelta(days=1)

human_experts = [
    HumanExpert(
        expert_id='E001',
        name='Dr. Sarah Chen',
        experience_years=25,
        expertise_areas=['dredging_operations', 'sediment_transport', 'environmental_impact'],
        trust_level=0.6,
        decision_accuracy=0.85,
        availability=0.8,
        feedback_quality=0.9,
    ),
    HumanExpert(
        expert_id='E002',
        name='Capt. Michael Rodriguez',
        experience_years=20,
        expertise_areas=['maritime_navigation', 'vessel_traffic', 'safety_operations'],
        trust_level=0.5,
        decision_accuracy=0.82,
        availability=0.7,
        feedback_quality=0.85,
    ),
    HumanExpert(
        expert_id='E003',
        name='Prof. James Wilson',
        experience_years=30,
        expertise_areas=['coastal_engineering', 'structural_analysis', 'economic_modeling'],
        trust_level=0.7,
        decision_accuracy=0.88,
        availability=0.6,
        feedback_quality=0.92,
    ),
]

segments_config = {
    0: {'name': 'Entrance Channel', 'complexity': 'high', 'risk_level': 0.7},
    1: {'name': 'Inner Channel', 'complexity': 'medium', 'risk_level': 0.5},
    2: {'name': 'Berth Approach', 'complexity': 'low', 'risk_level': 0.3},
}

print('Human–AI partnership configuration:')
print(f'- Partnership duration: {partnership_duration.days} days')
print(f'- Decision interval: {decision_interval.days} day(s)')
print(f'- Human experts: {len(human_experts)}')
print(f'- Channel segments: {len(segments_config)}')
print(f'- Collaboration periods: {int(partnership_duration / decision_interval)}')

In [ ]:
class ExplainableAI:
    """Rule-based recommendation system with explanatory text and confidence."""

    def __init__(self, segments_config: Dict[int, Dict[str, Any]]):
        self.segments_config = segments_config
        self.model_confidence_base = 0.75
        self.explanation_templates = {
            'depth_analysis': 'Channel depth analysis shows {depth:.2f} m current versus {target:.2f} m target',
            'traffic_impact': 'Current traffic volume of {traffic} vessels/day indicates {impact}',
            'weather_factor': 'Weather conditions ({weather}) affect operations by {effect}',
            'economic_benefit': 'Illustrative economic benefit of ${benefit:.1f}M with estimated ROI {roi:.1f}%',
            'risk_assessment': 'Risk level {risk} for this segment ({segment_risk:.1f} baseline risk score): {factors}',
        }

    def generate_recommendation(
        self,
        segment_id: int,
        current_conditions: Dict[str, Any],
    ) -> AIRecommendation:
        """Apply depth and traffic rules, then attach confidence and explanation."""
        if segment_id not in self.segments_config:
            raise KeyError(f'Unknown segment_id: {segment_id}')

        current_depth = current_conditions.get('depth', 11.6)
        target_depth = 14.0
        depth_deficit = target_depth - current_depth
        traffic_volume = current_conditions.get('traffic', 10)
        weather_condition = current_conditions.get('weather', 'clear')

        if depth_deficit > 1.0:
            action = 'immediate_dredging'
            decision_type = 'dredging'
        elif depth_deficit > 0.3:
            action = 'scheduled_dredging'
            decision_type = 'dredging'
        elif traffic_volume > 15:
            action = 'monitor_closely'
            decision_type = 'monitoring'
        else:
            action = 'maintain_status'
            decision_type = 'monitoring'

        confidence = self._calculate_confidence(current_conditions, action)
        reasoning = self._generate_reasoning(current_conditions, action, depth_deficit)

        uncertainty_factors = []
        if weather_condition == 'storm':
            uncertainty_factors.append('severe_weather_conditions')
        if traffic_volume > 20:
            uncertainty_factors.append('unusual_traffic_patterns')
        if abs(depth_deficit) < 0.1:
            uncertainty_factors.append('measurement_precision_limits')

        # Expected improvements and benefits cannot be negative when the channel
        # is already at or above the target depth.
        positive_deficit = max(0.0, depth_deficit)
        expected_outcome = {
            'depth_improvement': min(positive_deficit, 2.0),
            'traffic_efficiency': traffic_volume * 0.1,
            'economic_benefit': positive_deficit * traffic_volume * 0.5,
            'environmental_impact': positive_deficit * 0.2,
        }

        timestamp = current_conditions.get('timestamp', datetime.now())
        recommendation_id = f"AI_{timestamp:%Y%m%dT%H%M%S}_S{segment_id}"
        return AIRecommendation(
            recommendation_id=recommendation_id,
            timestamp=timestamp,
            decision_type=decision_type,
            segment_id=segment_id,
            action=action,
            confidence=confidence,
            reasoning=reasoning,
            data_sources=['depth_sensors', 'traffic_monitors', 'weather_stations', 'economic_models'],
            uncertainty_factors=uncertainty_factors,
            expected_outcome=expected_outcome,
        )

    def _calculate_confidence(self, conditions: Dict[str, Any], action: str) -> float:
        """Combine assumed data quality, action complexity, and historical accuracy."""
        data_quality = 1.0
        if conditions.get('sensor_status') != 'optimal':
            data_quality *= 0.9
        if conditions.get('weather') == 'fog':
            data_quality *= 0.8

        complexity_factor = {
            'immediate_dredging': 0.8,
            'scheduled_dredging': 0.9,
            'monitor_closely': 0.95,
            'maintain_status': 0.98,
        }.get(action, 0.9)
        historical_accuracy = 0.85
        confidence = self.model_confidence_base * data_quality * complexity_factor * historical_accuracy
        return max(0.3, min(0.98, confidence))

    def _generate_reasoning(
        self,
        conditions: Dict[str, Any],
        action: str,
        depth_deficit: float,
    ) -> List[str]:
        """Build a readable explanation from the same inputs used by the rules."""
        reasoning = []
        current_depth = conditions.get('depth', 11.6)
        reasoning.append(
            self.explanation_templates['depth_analysis'].format(
                depth=current_depth,
                target=14.0,
            )
        )

        traffic = conditions.get('traffic', 10)
        if traffic > 15:
            impact = 'high congestion risk'
        elif traffic > 10:
            impact = 'moderate traffic pressure'
        else:
            impact = 'normal traffic conditions'
        reasoning.append(
            self.explanation_templates['traffic_impact'].format(traffic=traffic, impact=impact)
        )

        weather = conditions.get('weather', 'clear')
        if weather == 'storm':
            effect = 'reduced visibility and increased operational risk'
        elif weather == 'fog':
            effect = 'navigation difficulties'
        else:
            effect = 'the conditions represented in this model'
        reasoning.append(
            self.explanation_templates['weather_factor'].format(weather=weather, effect=effect)
        )

        positive_deficit = max(0.0, depth_deficit)
        benefit = positive_deficit * traffic * 0.5
        roi = (benefit / (positive_deficit * 10)) * 100 if positive_deficit > 0 else 0.0
        reasoning.append(
            self.explanation_templates['economic_benefit'].format(benefit=benefit, roi=roi)
        )

        segment_risk = self.segments_config[conditions.get('segment_id', 0)]['risk_level']
        if depth_deficit > 1.0:
            risk = 'HIGH'
            factors = 'a significant depth deficit and the segment baseline risk'
        elif depth_deficit > 0.3:
            risk = 'MEDIUM'
            factors = 'a moderate depth deficit and the segment baseline risk'
        else:
            risk = 'LOW'
            factors = 'the modeled depth deficit and the segment baseline risk'
        reasoning.append(
            self.explanation_templates['risk_assessment'].format(
                risk=risk,
                segment_risk=segment_risk,
                factors=factors,
            )
        )
        return reasoning


class HumanAICollaboration:
    """Select experts, simulate their responses, and record trust measures."""

    def __init__(self, human_experts: List[HumanExpert], ai_system: ExplainableAI):
        if not human_experts:
            raise ValueError('At least one human expert is required.')
        self.human_experts = human_experts
        self.ai_system = ai_system
        self.collaboration_history = []
        self.trust_metrics = []
        self.learning_data = []

    def select_expert_for_decision(self, recommendation: AIRecommendation) -> HumanExpert:
        """Rank experts by availability, trust, and decision-type expertise."""
        expert_scores: List[Tuple[HumanExpert, float]] = []
        for expert in self.human_experts:
            expertise_score = 0.5
            if recommendation.decision_type == 'dredging':
                if 'dredging_operations' in expert.expertise_areas:
                    expertise_score = 1.0
                elif 'sediment_transport' in expert.expertise_areas:
                    expertise_score = 0.8
            elif recommendation.decision_type == 'monitoring':
                if 'vessel_traffic' in expert.expertise_areas:
                    expertise_score = 1.0
                elif 'safety_operations' in expert.expertise_areas:
                    expertise_score = 0.8

            overall_score = (
                expert.availability * 0.3
                + expert.trust_level * 0.2
                + expertise_score * 0.5
            )
            expert_scores.append((expert, overall_score))

        return max(expert_scores, key=lambda item: item[1])[0]

    def simulate_human_decision(
        self,
        recommendation: AIRecommendation,
        expert: HumanExpert,
    ) -> HumanDecision:
        """Sample a response using recommendation confidence and expert trust."""
        acceptance_probability = recommendation.confidence * expert.trust_level
        expertise_factor = 1.0
        if recommendation.decision_type == 'dredging':
            if 'dredging_operations' in expert.expertise_areas:
                expertise_factor = 1.2
        elif recommendation.decision_type == 'monitoring':
            if 'vessel_traffic' in expert.expertise_areas:
                expertise_factor = 1.1
        acceptance_probability = min(1.0, acceptance_probability * expertise_factor)

        random_value = np.random.random()
        if random_value < acceptance_probability * 0.7:
            action = 'accept'
        elif random_value < acceptance_probability * 0.9:
            action = 'modify'
        else:
            action = 'reject'

        reasoning = self._generate_human_reasoning(recommendation, expert, action)
        human_confidence = expert.decision_accuracy
        if action == 'accept':
            human_confidence *= 1.1
        elif action == 'reject':
            human_confidence *= 0.9

        time_spent = int(5 + np.random.exponential(10))
        if action == 'modify':
            time_spent = int(time_spent * 1.5)

        decision_id = f"HD_{recommendation.timestamp:%Y%m%dT%H%M%S}_S{recommendation.segment_id}"
        return HumanDecision(
            decision_id=decision_id,
            expert_id=expert.expert_id,
            recommendation_id=recommendation.recommendation_id,
            timestamp=recommendation.timestamp,
            action=action,
            reasoning=reasoning,
            confidence=min(1.0, human_confidence),
            time_spent_minutes=time_spent,
            additional_context={
                'expertise_used': expert.expertise_areas[0],
                'experience_applied': f'{expert.experience_years} years',
                'trust_level_at_decision': expert.trust_level,
            },
        )

    def _generate_human_reasoning(
        self,
        recommendation: AIRecommendation,
        expert: HumanExpert,
        action: str,
    ) -> str:
        """Choose an illustrative rationale appropriate to the response."""
        templates = {
            'accept': [
                'AI analysis aligns with my experience and current conditions',
                'The recommendation is supported by the presented data and reasoning',
                'The proposed timing is appropriate for the modeled operational context',
                'The expected outcomes are plausible within the assumptions of this model',
            ],
            'modify': [
                'The recommendation needs adjustment for local conditions',
                'The analysis is useful, but implementation timing should be modified',
                'Additional operational factors should be considered',
                'The action should be scaled to reflect resource availability',
            ],
            'reject': [
                'The analysis does not represent important local knowledge',
                'The recommendation conflicts with recent operational experience',
                'The modeled conditions call for a different approach',
                'The risk assessment does not reflect the expert’s view of current challenges',
            ],
        }
        return str(np.random.choice(templates[action]))

    def update_trust_metrics(
        self,
        decision: HumanDecision,
        recommendation: AIRecommendation,
    ) -> None:
        """Apply the specified bounded trust update and record interaction proxies."""
        expert = next(
            (item for item in self.human_experts if item.expert_id == decision.expert_id),
            None,
        )
        if expert is None:
            raise ValueError(f'Unknown expert_id: {decision.expert_id}')

        if decision.action == 'accept':
            expert.trust_level = min(1.0, expert.trust_level + 0.02)
        elif decision.action == 'reject':
            expert.trust_level = max(0.1, expert.trust_level - 0.01)

        metrics = TrustMetrics(
            timestamp=decision.timestamp,
            expert_id=decision.expert_id,
            ai_trust_score=expert.trust_level,
            recommendation_trust=recommendation.confidence,
            explanation_trust=0.8 if len(recommendation.reasoning) > 3 else 0.6,
            overall_satisfaction=0.85 if decision.action == 'accept' else 0.6,
            collaboration_effectiveness=0.9 if decision.time_spent_minutes < 15 else 0.7,
        )
        self.trust_metrics.append(metrics)


print('Explainable AI and human–AI collaboration classes are defined.')

In [ ]:
class HumanAIPartnership:
    """Run the sequential partnership simulation and summarize its indicators."""

    def __init__(
        self,
        human_experts: List[HumanExpert],
        segments_config: Dict[int, Dict[str, Any]],
        partnership_start: datetime = partnership_start,
        partnership_duration: timedelta = partnership_duration,
        decision_interval: timedelta = decision_interval,
    ):
        if decision_interval.total_seconds() <= 0:
            raise ValueError('decision_interval must be positive.')
        if partnership_duration.total_seconds() < 0:
            raise ValueError('partnership_duration cannot be negative.')
        if not segments_config:
            raise ValueError('At least one channel segment is required.')

        self.human_experts = human_experts
        self.segments_config = segments_config
        self.ai_system = ExplainableAI(segments_config)
        self.collaboration = HumanAICollaboration(human_experts, self.ai_system)
        self.partnership_start = partnership_start
        self.partnership_duration = partnership_duration
        self.decision_interval = decision_interval
        self.current_time = partnership_start

        self.initial_trust_levels = {
            expert.expert_id: expert.trust_level for expert in human_experts
        }
        self.recommendations: List[AIRecommendation] = []
        self.decisions: List[HumanDecision] = []
        self.trust_evolution = []
        self.learning_insights = []
        self.collaboration_efficiency = []
        self.decision_quality = []
        self.trust_development = []

    def simulate_partnership_day(self) -> Dict[str, Any]:
        """Simulate recommendations and human reviews for one partnership day."""
        daily_results = {
            'date': self.current_time.date(),
            'recommendations': [],
            'decisions': [],
            'trust_changes': [],
            'collaboration_metrics': {},
        }

        for segment_id in self.segments_config:
            conditions = self._generate_current_conditions(segment_id)
            recommendation = self.ai_system.generate_recommendation(segment_id, conditions)
            self.recommendations.append(recommendation)
            daily_results['recommendations'].append(recommendation)

            expert = self.collaboration.select_expert_for_decision(recommendation)
            decision = self.collaboration.simulate_human_decision(recommendation, expert)
            self.decisions.append(decision)
            daily_results['decisions'].append(decision)

            old_trust = expert.trust_level
            self.collaboration.update_trust_metrics(decision, recommendation)
            new_trust = expert.trust_level
            daily_results['trust_changes'].append(
                {
                    'expert_id': expert.expert_id,
                    'old_trust': old_trust,
                    'new_trust': new_trust,
                    'change': new_trust - old_trust,
                }
            )

        daily_results['collaboration_metrics'] = self._calculate_daily_metrics(daily_results)
        self.current_time += self.decision_interval
        return daily_results

    def _generate_current_conditions(self, segment_id: int) -> Dict[str, Any]:
        """Sample illustrative operating conditions for a segment and date."""
        base_depth = 11.6 + np.random.normal(0, 0.5)
        base_traffic = 10 + np.random.poisson(5)
        weather_options = ['clear', 'cloudy', 'rainy', 'fog', 'storm']
        weather_probabilities = [0.4, 0.3, 0.15, 0.1, 0.05]
        weather = np.random.choice(weather_options, p=weather_probabilities)
        sensor_status = 'optimal' if np.random.random() > 0.1 else 'degraded'

        return {
            'segment_id': segment_id,
            'depth': max(10.5, base_depth),
            'traffic': max(0, base_traffic),
            'weather': weather,
            'sensor_status': sensor_status,
            'timestamp': self.current_time,
        }

    def _calculate_daily_metrics(self, daily_results: Dict[str, Any]) -> Dict[str, float]:
        """Calculate response, confidence, time, trust, and heuristic efficiency metrics."""
        decisions = daily_results['decisions']
        recommendations = daily_results['recommendations']
        trust_changes = daily_results['trust_changes']
        if not decisions or not recommendations:
            raise ValueError('Daily metrics require at least one decision and recommendation.')

        decision_count = len(decisions)
        accept_rate = sum(decision.action == 'accept' for decision in decisions) / decision_count
        modify_rate = sum(decision.action == 'modify' for decision in decisions) / decision_count
        reject_rate = sum(decision.action == 'reject' for decision in decisions) / decision_count
        avg_ai_confidence = float(np.mean([item.confidence for item in recommendations]))
        avg_human_confidence = float(np.mean([item.confidence for item in decisions]))
        avg_trust_change = float(np.mean([item['change'] for item in trust_changes]))
        avg_decision_time = float(np.mean([item.time_spent_minutes for item in decisions]))
        avg_explanation_length = float(np.mean([len(item.reasoning) for item in recommendations]))
        avg_trust_level = float(np.mean([expert.trust_level for expert in self.human_experts]))

        # This bounded score is a time-based indicator, not a measured productivity rate.
        collaboration_efficiency = float(np.clip(1.0 - (avg_decision_time - 10) / 30, 0.0, 1.0))
        return {
            'acceptance_rate': accept_rate,
            'modification_rate': modify_rate,
            'rejection_rate': reject_rate,
            'avg_ai_confidence': avg_ai_confidence,
            'avg_human_confidence': avg_human_confidence,
            'avg_trust_change': avg_trust_change,
            'avg_trust_level': avg_trust_level,
            'avg_decision_time_minutes': avg_decision_time,
            'avg_explanation_length': avg_explanation_length,
            'collaboration_efficiency': collaboration_efficiency,
        }

    def run_partnership_simulation(self) -> Dict[str, Any]:
        """Run the partnership over its configured duration and return the records."""
        print('=' * 70)
        print('HUMAN–AI SYMBIOTIC PARTNERSHIP SIMULATION')
        print('=' * 70)
        print(f'\nPartnership duration: {self.partnership_duration.days} days')
        print(f'Human experts: {len(self.human_experts)}')
        print(f'Channel segments: {len(self.segments_config)}')
        print(f'Decisions per day: {len(self.segments_config)}')

        total_days = int(self.partnership_duration / self.decision_interval)
        partnership_results = []
        for day in range(total_days):
            daily_result = self.simulate_partnership_day()
            partnership_results.append(daily_result)

            if (day + 1) % 30 == 0 or day == total_days - 1:
                metrics = daily_result['collaboration_metrics']
                print(
                    f"Day {day + 1:3d}: {daily_result['date']:%Y-%m-%d} - "
                    f"Accept: {metrics['acceptance_rate']:.1%}, "
                    f"Mean trust: {metrics['avg_trust_level']:.3f}, "
                    f"Time-based efficiency: {metrics['collaboration_efficiency']:.2f}"
                )

        print(f'\nSimulation completed: {len(partnership_results)} days')
        analysis = self._analyze_partnership_results(partnership_results)
        return {
            'partnership_results': partnership_results,
            'analysis': analysis,
            'final_trust_levels': {
                expert.expert_id: expert.trust_level for expert in self.human_experts
            },
            'total_decisions': len(self.decisions),
            'total_recommendations': len(self.recommendations),
        }

    def _analyze_partnership_results(self, results: List[Dict[str, Any]]) -> Dict[str, Any]:
        """Aggregate daily metrics and compare trust with its recorded initial state."""
        dates = [result['date'] for result in results]
        acceptance_rates = [result['collaboration_metrics']['acceptance_rate'] for result in results]
        trust_changes = [result['collaboration_metrics']['avg_trust_change'] for result in results]
        efficiencies = [result['collaboration_metrics']['collaboration_efficiency'] for result in results]
        avg_trust_levels = [result['collaboration_metrics']['avg_trust_level'] for result in results]

        trust_evolution = {}
        for expert in self.human_experts:
            initial = self.initial_trust_levels[expert.expert_id]
            trust_evolution[expert.expert_id] = {
                'initial': initial,
                'final': expert.trust_level,
                'improvement': expert.trust_level - initial,
            }

        initial_mean_trust = float(np.mean(list(self.initial_trust_levels.values())))
        final_mean_trust = float(np.mean([expert.trust_level for expert in self.human_experts]))
        decision_quality = self._analyze_decision_quality()
        return {
            'dates': dates,
            'acceptance_rates': acceptance_rates,
            'trust_changes': trust_changes,
            'avg_trust_levels': avg_trust_levels,
            'efficiencies': efficiencies,
            'overall_acceptance': float(np.mean(acceptance_rates)) if acceptance_rates else 0.0,
            'overall_efficiency': float(np.mean(efficiencies)) if efficiencies else 0.0,
            'trust_improvement': final_mean_trust - initial_mean_trust,
            'trust_evolution': trust_evolution,
            'decision_quality': decision_quality,
        }

    def _analyze_decision_quality(self) -> Dict[str, float]:
        """Apply assumed response-specific success rates as a scenario indicator."""
        if not self.decisions:
            return {'overall_quality': 0.0}

        success_rates = {'accept': 0.85, 'modify': 0.90, 'reject': 0.80}
        total_decisions = len(self.decisions)
        overall_quality = sum(
            sum(decision.action == action for decision in self.decisions) * rate
            for action, rate in success_rates.items()
        ) / total_decisions
        return {
            'overall_quality': overall_quality,
            'accept_success_rate': success_rates['accept'],
            'modify_success_rate': success_rates['modify'],
            'reject_success_rate': success_rates['reject'],
            'human_improvement_factor': 0.05,
        }


print('Human–AI partnership simulation class is defined.')

In [ ]:
# The simulation uses NumPy's global random generator, as do its condition and response rules.
partnership = HumanAIPartnership(human_experts, segments_config)
partnership_results = partnership.run_partnership_simulation()
analysis = partnership_results['analysis']

print('\n' + '=' * 60)
print('PARTNERSHIP SIMULATION SUMMARY')
print('=' * 60)
print(f"Total recommendations: {partnership_results['total_recommendations']}")
print(f"Total decisions: {partnership_results['total_decisions']}")
print(f"Mean daily acceptance rate: {analysis['overall_acceptance']:.1%}")
print(f"Mean time-based efficiency indicator: {analysis['overall_efficiency']:.2f}")
print(f"Change in mean expert trust: {analysis['trust_improvement']:+.3f}")
print(
    'Assumption-based decision-quality indicator: '
    f"{analysis['decision_quality']['overall_quality']:.1%}"
)

print('\nFinal trust levels by expert:')
for expert in human_experts:
    trust_data = analysis['trust_evolution'][expert.expert_id]
    print(
        f"- {expert.name}: {trust_data['final']:.3f} "
        f"(change: {trust_data['improvement']:+.3f})"
    )

print('\nAssumed success rates used in the scenario indicator:')
quality = analysis['decision_quality']
for action in ['accept', 'modify', 'reject']:
    print(f"- {action.title()}: {quality[f'{action}_success_rate']:.1%}")
print(f"- Assumed human improvement factor: {quality['human_improvement_factor']:.1%}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Human–AI Partnership Simulation Indicators', fontsize=16, fontweight='bold')
days = [(date - partnership_start.date()).days for date in analysis['dates']]

ax = axes[0, 0]
ax.plot(days, np.array(analysis['acceptance_rates']) * 100, linewidth=2, color='blue')
ax.fill_between(days, np.array(analysis['acceptance_rates']) * 100, alpha=0.25, color='lightblue')
ax.set_xlabel('Partnership day')
ax.set_ylabel('Daily acceptance rate (%)')
ax.set_title('AI Recommendation Acceptance')
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(days, analysis['avg_trust_levels'], linewidth=2, color='green')
ax.set_xlabel('Partnership day')
ax.set_ylabel('Mean expert trust (0–1)')
ax.set_title('Mean Trust Across Experts')
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

ax = axes[0, 2]
ax.plot(days, np.array(analysis['efficiencies']) * 100, linewidth=2, color='purple')
ax.set_xlabel('Partnership day')
ax.set_ylabel('Time-based efficiency indicator (%)')
ax.set_title('Decision-Time Indicator')
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
expert_names = [expert.name.split()[-1] for expert in human_experts]
initial_trusts = [analysis['trust_evolution'][expert.expert_id]['initial'] for expert in human_experts]
final_trusts = [analysis['trust_evolution'][expert.expert_id]['final'] for expert in human_experts]
positions = np.arange(len(expert_names))
width = 0.35
ax.bar(positions - width / 2, initial_trusts, width, label='Initial trust', color='lightcoral', edgecolor='black')
ax.bar(positions + width / 2, final_trusts, width, label='Final trust', color='lightblue', edgecolor='black')
ax.set_xlabel('Human expert')
ax.set_ylabel('Trust level (0–1)')
ax.set_title('Initial and Final Trust by Expert')
ax.set_xticks(positions)
ax.set_xticklabels(expert_names, rotation=30, ha='right')
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, axis='y', alpha=0.3)

ax = axes[1, 1]
decision_types = ['Accept', 'Modify', 'Reject']
success_rates = [
    analysis['decision_quality']['accept_success_rate'],
    analysis['decision_quality']['modify_success_rate'],
    analysis['decision_quality']['reject_success_rate'],
]
bars = ax.bar(decision_types, np.array(success_rates) * 100,
              color=['lightgreen', 'gold', 'lightcoral'], edgecolor='black', alpha=0.85)
ax.set_ylabel('Assumed success rate (%)')
ax.set_title('Scenario Assumptions by Response')
ax.set_ylim(0, 100)
ax.grid(True, axis='y', alpha=0.3)
for bar, rate in zip(bars, success_rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{rate:.0%}', ha='center', va='bottom')

ax = axes[1, 2]
summary_labels = ['Acceptance', 'Time-based\nefficiency', 'Assumption-based\nquality']
summary_values = [
    analysis['overall_acceptance'],
    analysis['overall_efficiency'],
    analysis['decision_quality']['overall_quality'],
]
bars = ax.bar(summary_labels, np.array(summary_values) * 100,
              color=plt.cm.Set3(np.linspace(0, 1, len(summary_labels))), edgecolor='black')
ax.set_ylabel('Indicator value (%)')
ax.set_title('Summary of Distinct Indicators')
ax.set_ylim(0, 100)
ax.tick_params(axis='x', rotation=15)
ax.grid(True, axis='y', alpha=0.3)
for bar, value in zip(bars, summary_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{value:.1%}', ha='center', va='bottom')

fig.tight_layout()
plt.show()

print('\nThe panels show daily response and trust metrics, a time-based indicator,')
print('expert trust changes, assumed response success rates, and a compact metric summary.')
print('The indicators have different definitions and should not be interpreted as interchangeable outcomes.')

In [ ]:
def analyze_ai_learning(partnership: HumanAIPartnership) -> List[Dict[str, Any]]:
    """Summarize recorded feedback; this function does not train or update the AI."""
    recommendations_by_id = {
        recommendation.recommendation_id: recommendation
        for recommendation in partnership.recommendations
    }
    learning_data = []
    for decision in partnership.decisions:
        recommendation = recommendations_by_id.get(decision.recommendation_id)
        if recommendation is None:
            continue
        learning_data.append(
            {
                'day': (decision.timestamp.date() - partnership.partnership_start.date()).days,
                'ai_confidence': recommendation.confidence,
                'human_action': decision.action,
                'human_confidence': decision.confidence,
                'expertise': decision.additional_context.get('expertise_used', 'unknown'),
                'decision_time': decision.time_spent_minutes,
            }
        )
    return learning_data


learning_data = analyze_ai_learning(partnership)
print('\n' + '=' * 60)
print('RECORDED FEEDBACK AND EXPERTISE ANALYSIS')
print('=' * 60)

if learning_data:
    ai_confidences = [item['ai_confidence'] for item in learning_data]
    human_confidences = [item['human_confidence'] for item in learning_data]
    groups = {
        action: [item for item in learning_data if item['human_action'] == action]
        for action in ['accept', 'modify', 'reject']
    }

    print('Recorded interaction summary:')
    print(f'- Feedback records: {len(learning_data)}')
    print(f'- First recorded AI confidence: {ai_confidences[0]:.3f}')
    print(f'- Last recorded AI confidence: {ai_confidences[-1]:.3f}')
    print('- AI confidence is calculated from fixed rules; this difference is not model learning.')
    print(f'- Mean human confidence: {np.mean(human_confidences):.3f}')

    print('\nResponse counts:')
    for action, records in groups.items():
        print(f'- {action.title()}: {len(records)} ({len(records) / len(learning_data):.1%})')
        if records:
            mean_confidence = np.mean([item['ai_confidence'] for item in records])
            print(f'  Mean AI confidence for these recommendations: {mean_confidence:.3f}')
else:
    print('No interaction records are available.')

print('\nExpert contribution summary:')
for expert in human_experts:
    expert_decisions = [item for item in partnership.decisions if item.expert_id == expert.expert_id]
    if not expert_decisions:
        print(f'- {expert.name}: no recorded decisions')
        continue

    mean_confidence = np.mean([item.confidence for item in expert_decisions])
    mean_time = np.mean([item.time_spent_minutes for item in expert_decisions])
    acceptance_rate = sum(item.action == 'accept' for item in expert_decisions) / len(expert_decisions)
    trust_change = analysis['trust_evolution'][expert.expert_id]['improvement']
    print(f'- {expert.name}: {len(expert_decisions)} decisions, acceptance {acceptance_rate:.1%}')
    print(f'  Mean confidence: {mean_confidence:.3f}; mean decision time: {mean_time:.1f} minutes')
    print(f'  Simulated trust change: {trust_change:+.3f}')

print('\nInterpretation note:')
print('These summaries describe simulated interactions. They do not establish learning gains,')
print('return on investment, or causal improvement in decision quality.')

In [ ]:
def safe_correlation(first: List[float], second: List[float]) -> Optional[float]:
    """Return Pearson correlation only when both inputs vary and have enough observations."""
    if len(first) < 2 or len(second) < 2 or len(first) != len(second):
        return None
    if np.std(first) == 0 or np.std(second) == 0:
        return None
    correlation = float(np.corrcoef(first, second)[0, 1])
    return correlation if np.isfinite(correlation) else None


def analyze_explanation_quality(
    recommendations: List[AIRecommendation],
) -> Dict[str, List[float]]:
    """Describe explanation structure, source count, uncertainty disclosure, and confidence."""
    metrics = {
        'explanation_length': [],
        'reasoning_categories': [],
        'confidence': [],
        'data_source_diversity': [],
        'uncertainty_transparency': [],
    }
    for recommendation in recommendations:
        metrics['explanation_length'].append(len(recommendation.reasoning))
        metrics['data_source_diversity'].append(len(recommendation.data_sources))
        metrics['uncertainty_transparency'].append(len(recommendation.uncertainty_factors))
        metrics['confidence'].append(recommendation.confidence)

        categories = set()
        for reason in recommendation.reasoning:
            text = reason.lower()
            if 'depth' in text:
                categories.add('depth analysis')
            if 'traffic' in text:
                categories.add('traffic analysis')
            if 'weather' in text:
                categories.add('weather analysis')
            if 'economic' in text:
                categories.add('economic analysis')
            if 'risk' in text:
                categories.add('risk assessment')
        metrics['reasoning_categories'].append(len(categories))
    return metrics


explanation_analysis = analyze_explanation_quality(partnership.recommendations)
print('\n' + '=' * 60)
print('EXPLANATION STRUCTURE AND HUMAN RESPONSE')
print('=' * 60)

if explanation_analysis['explanation_length']:
    print('AI explanation structure:')
    print(f"- Mean explanation length: {np.mean(explanation_analysis['explanation_length']):.1f} reasoning steps")
    print(f"- Mean reasoning categories: {np.mean(explanation_analysis['reasoning_categories']):.1f}")
    print(f"- Mean listed data sources: {np.mean(explanation_analysis['data_source_diversity']):.1f}")
    print(f"- Mean listed uncertainty factors: {np.mean(explanation_analysis['uncertainty_transparency']):.1f}")
    explanation_confidence_correlation = safe_correlation(
        explanation_analysis['confidence'],
        explanation_analysis['explanation_length'],
    )
    if explanation_confidence_correlation is None:
        print('- Confidence and explanation length correlation: not estimable from these data')
    else:
        print(f'- Confidence and explanation length correlation: {explanation_confidence_correlation:.3f}')
else:
    print('No explanation records are available.')

decision_times = [decision.time_spent_minutes for decision in partnership.decisions]
human_confidences = [decision.confidence for decision in partnership.decisions]
print('\nHuman decision indicators:')
if decision_times:
    print(f'- Mean decision time: {np.mean(decision_times):.1f} minutes')
    for action in ['accept', 'modify', 'reject']:
        action_times = [
            decision.time_spent_minutes
            for decision in partnership.decisions
            if decision.action == action
        ]
        if action_times:
            print(f'- Mean time for {action} decisions: {np.mean(action_times):.1f} minutes')
    print(f'- Mean human confidence: {np.mean(human_confidences):.3f}')
    print(f'- Human confidence standard deviation: {np.std(human_confidences):.3f}')
    high_count = sum(confidence > 0.9 for confidence in human_confidences)
    low_count = sum(confidence < 0.7 for confidence in human_confidences)
    print(f'- High-confidence decisions (>0.9): {high_count}/{len(human_confidences)}')
    print(f'- Lower-confidence decisions (<0.7): {low_count}/{len(human_confidences)}')
else:
    print('No human decisions are available.')

high_trust_decisions = []
low_trust_decisions = []
for decision in partnership.decisions:
    trust_at_decision = decision.additional_context['trust_level_at_decision']
    if trust_at_decision > 0.7:
        high_trust_decisions.append(decision)
    else:
        low_trust_decisions.append(decision)

print('\nResponse patterns by trust at decision time:')
for label, decisions in [('Higher trust (>0.7)', high_trust_decisions), ('Trust at or below 0.7', low_trust_decisions)]:
    if decisions:
        acceptance_rate = sum(decision.action == 'accept' for decision in decisions) / len(decisions)
        print(f'- {label}: acceptance rate {acceptance_rate:.1%} across {len(decisions)} decisions')
    else:
        print(f'- {label}: no decisions in this group')

recommendations_by_id = {
    recommendation.recommendation_id: recommendation
    for recommendation in partnership.recommendations
}
explanation_lengths = []
decision_confidences = []
for decision in partnership.decisions:
    recommendation = recommendations_by_id.get(decision.recommendation_id)
    if recommendation is not None:
        explanation_lengths.append(len(recommendation.reasoning))
        decision_confidences.append(decision.confidence)

correlation = safe_correlation(explanation_lengths, decision_confidences)
print('\nExplanation length and human confidence:')
if correlation is None:
    print('- Correlation is not estimable because the sample is too small or a measure does not vary.')
else:
    print(f'- Pearson correlation: {correlation:.3f}')
    print('- This descriptive association does not establish that explanation length causes confidence.')

print('\nModel interpretation:')
print('- Explanation length is a structural count, not a direct measure of clarity or understanding.')
print('- Confidence and trust are simulated quantities and require empirical validation for operational use.')

## Summary and Conclusions

### What this implementation demonstrates

The notebook implements a sequential human–AI partnership simulation. For each simulated segment-day, a rule-based system recommends an action and provides a short explanation; a selected expert accepts, modifies, or rejects the proposal; and the expert's trust is updated according to the stated rule. The records support descriptive summaries of responses, confidence, decision time, and trust.

### How to interpret the indicators

- **Acceptance, modification, and rejection rates** describe the simulated responses to recommendations. Acceptance is not evidence that a recommendation is correct.
- **Trust levels** follow the assumed response-based update rule. They are not psychological measurements.
- **The time-based efficiency indicator** is a bounded transformation of simulated decision time. It is not a measured productivity rate or a cost-benefit calculation.
- **The decision-quality indicator** uses assumed response-specific success rates. It is a scenario calculation, not an observed success rate.
- **Explanation counts and correlations** describe properties of the generated text and their association with simulated confidence. They do not establish explanation quality, comprehension, or causality.

### Method and limitations

The central method is a stochastic, sequential simulation with rule-based recommendations, expert scoring, random response sampling, and bounded trust updates. Trust is the changing state that links one interaction to later decisions. The work per simulated day grows with the number of segments and experts: evaluating expert selection takes time proportional to the number of experts for each segment, while condition generation and response sampling are performed once per segment.

The AI's rules and confidence calculation remain fixed throughout the run. Logging feedback enables analysis but does not itself constitute learning. Conditions are synthetic, and the notebook does not validate recommendation outcomes against operational observations. It also does not calculate dredging investment costs, channel capacity, environmental consequences, or project returns. Therefore, its outputs should be read as consequences of the specified simulation assumptions, not as investment advice or evidence that collaboration outperforms either human-only or AI-only decision-making.

### Natural extensions

A fuller study could replace assumed condition distributions and response rules with validated operational data, record realized outcomes, and define explicit investment costs and constraints. Such extensions would require new evidence and modeling assumptions; they are outside the simulation implemented here. Any claim about learning, safety, economic benefit, or comparative decision quality would require an appropriate evaluation design and measured outcomes.